# Módulo 5 | SQL: Agregações




In [ ]:
# M5 — SQL: Agregações
# Sem AWS, sem Athena e sem upload manual

import pandas as pd
import numpy as np
import sqlite3

print("=" * 70)
print("M5 — SQL: AGREGAÇÕES | EBAC")
print("Notebook corrigido para Colab")
print("=" * 70)


# 1. Criar/carregar a tabela heartattack

# Estrutura usada no exercício:
COLUNAS = [
    'age', 'sex', 'cp', 'trtbps', 'chol', 'fbs', 'restecg',
    'thalachh', 'exng', 'oldpeak', 'slp', 'caa', 'thall', 'output'
]

# Tentativas de leitura de bases públicas com a mesma estrutura.
# Se uma URL falhar, o código segue para a próxima.
urls = [
    'https://raw.githubusercontent.com/Apaulgithub/oibsip_taskno1/main/heart.csv',
    'https://raw.githubusercontent.com/siddharth110/Heart-Disease-Prediction/master/heart.csv',
    'https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv'
]

df = None
for url in urls:
    try:
        temp = pd.read_csv(url)
        temp.columns = [str(c).strip().lower() for c in temp.columns]
        # Aceita exatamente as colunas do exercício.
        if set(COLUNAS).issubset(set(temp.columns)):
            df = temp[COLUNAS].copy()
            print(f" Base carregada da web: {url}")
            break
    except Exception as e:
        pass

# Fallback 100% offline: cria uma base compatível com a estrutura do exercício.
if df is None:
    print("Não foi possível carregar a base pública. Usando base local simulada compatível.")
    np.random.seed(42)
    n = 303
    df = pd.DataFrame({
        'age':       np.random.randint(29, 78, n),
        'sex':       np.random.choice([0, 1], n, p=[0.32, 0.68]),
        'cp':        np.random.randint(0, 4, n),
        'trtbps':    np.random.randint(94, 201, n),
        'chol':      np.random.randint(126, 565, n),
        'fbs':       np.random.choice([0, 1], n, p=[0.85, 0.15]),
        'restecg':   np.random.randint(0, 3, n),
        'thalachh':  np.random.randint(71, 203, n),
        'exng':      np.random.choice([0, 1], n, p=[0.67, 0.33]),
        'oldpeak':   np.round(np.random.uniform(0, 6.2, n), 1),
        'slp':       np.random.randint(0, 3, n),
        'caa':       np.random.randint(0, 5, n),
        'thall':     np.random.randint(0, 4, n),
        'output':    np.random.choice([0, 1], n, p=[0.46, 0.54])
    })

# Garante os tipos corretos para SQL.
for c in ['age','sex','cp','trtbps','chol','fbs','restecg','thalachh','exng','slp','caa','thall','output']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
df['oldpeak'] = pd.to_numeric(df['oldpeak'], errors='coerce').fillna(0).astype(float)

print(f"\n Dataset pronto: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(df.head(10).to_string(index=False))


# 2. Carregar no SQLite, simulando as consultas SQL do Athena

con = sqlite3.connect(':memory:')
df.to_sql('heartattack', con, index=False, if_exists='replace')
print("\n Tabela SQL 'heartattack' criada com sucesso.")

# Função auxiliar para executar, exibir e salvar cada query.
def executar_query(numero, titulo, sql):
    print("\n" + "=" * 70)
    print(f"QUERY {numero} — {titulo}")
    print("=" * 70)
    print(sql.strip())
    resultado = pd.read_sql_query(sql, con)
    print("\nResultado:")
    print(resultado.to_string(index=False))
    nome_arquivo = f"query_{numero}.csv"
    resultado.to_csv(nome_arquivo, index=False)
    print(f"\n {nome_arquivo} salvo")
    return resultado


# 3. Queries do exercício


query_1 = executar_query(
    1,
    "SELECT * FROM heartattack LIMIT 10",
    """
    SELECT *
    FROM heartattack
    LIMIT 10;
    """
)

query_2 = executar_query(
    2,
    "COUNT(age)",
    """
    SELECT COUNT(age) AS QUANTIDADE_LINHAS
    FROM heartattack;
    """
)

query_3 = executar_query(
    3,
    "COUNT por output com CASE",
    """
    SELECT
        COUNT(age) AS QUANTIDADE,
        CASE
            WHEN output = 1 THEN 'more chance of heart attack'
            ELSE 'less chance of heart attack'
        END AS output
    FROM heartattack
    GROUP BY output;
    """
)

query_4 = executar_query(
    4,
    "MAX, MIN e AVG de idade por output",
    """
    SELECT
        MAX(age) AS maior_idade,
        MIN(age) AS menor_idade,
        ROUND(AVG(age), 2) AS media_idade,
        output
    FROM heartattack
    GROUP BY output;
    """
)

query_5 = executar_query(
    5,
    "MAX, MIN e AVG de idade por output e sex",
    """
    SELECT
        MAX(age) AS maior_idade,
        MIN(age) AS menor_idade,
        ROUND(AVG(age), 2) AS media_idade,
        output,
        sex
    FROM heartattack
    GROUP BY output, sex;
    """
)

query_6 = executar_query(
    6,
    "HAVING COUNT(output) > 25",
    """
    SELECT
        COUNT(output) AS quantidade,
        output,
        sex
    FROM heartattack
    GROUP BY output, sex
    HAVING COUNT(output) > 25;
    """
)


# 4. Resumo final

print("\n" + "=" * 70)
print("RESUMO FINAL")
print("=" * 70)
for i, q in enumerate([query_1, query_2, query_3, query_4, query_5, query_6], start=1):
    print(f"query_{i}.csv → {len(q)} linhas")



M5 — SQL: AGREGAÇÕES | EBAC
Notebook corrigido para Colab
Não foi possível carregar a base pública. Usando base local simulada compatível.

 Dataset pronto: 303 linhas × 14 colunas
 age  sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  caa  thall  output
  67    0   3     152   302    0        0        85     1      0.1    2    4      0       0
  57    1   0     163   410    0        0       138     0      3.5    1    4      1       1
  43    0   2     194   188    0        2        91     1      3.3    0    2      0       0
  71    0   0     126   275    0        1        89     0      4.5    0    4      0       0
  36    1   1     146   279    0        2       104     0      5.5    1    4      1       0
  49    1   1     115   281    0        2       148     1      0.5    0    0      1       1
  67    1   3     114   495    0        1       195     1      4.5    0    3      3       0
  47    0   1     163   174    0        0       187     0      1.2    0    2      0